# 02f. Aplicar Splink 5 no Spark — modelo Luis

Predict com `splink_model_mae_endereco_luis` nos CSVs de aplicação do
[`07_exportar_csv_aplicacao_splink.ipynb`](07_exportar_csv_aplicacao_splink.ipynb),
via Spark + Splink 5 (sem `config.py` / DuckDB do projeto).

Paths em `/data/spark/shared/datasets/`:
- `censo_aplicacao_splink.csv` / `cpf_aplicacao_splink.csv` (sem header)
- `splink_model_mae_endereco_luis.json`
- saída: `splink_predictions_mae_endereco_luis.parquet`

Mesmas 14 regras de blocking do
[`02d_aplicar_splink_mae_endereco.ipynb`](02d_aplicar_splink_mae_endereco.ipynb);
`predict` com `threshold_match_probability=0.05`.


In [ ]:
from pathlib import Path

import json

from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import IntegerType, StringType, StructField, StructType

DATA_DIR = Path('/data/spark/shared/datasets')
CENSO_CSV = DATA_DIR / 'censo_aplicacao_splink.csv'
CPF_CSV = DATA_DIR / 'cpf_aplicacao_splink.csv'
MODEL_JSON = DATA_DIR / 'splink_model_mae_endereco_luis.json'
OUT_PARQUET = DATA_DIR / 'splink_predictions_mae_endereco_luis.parquet'

COLS = [
    'id_seq',
    'nome_completo_phon',
    'primeiro_nome_phon',
    'ultimo_nome_phon',
    'data_nascimento',
    'ano_nascimento',
    'mes_nascimento',
    'dia_nascimento',
    'idade',
    'nome_mae_phon',
    'logradouro_norm',
    'cep',
    'cod_municipio',
    'sexo',
]

schema = StructType([StructField(c, StringType(), True) for c in COLS])

spark = SparkSession.builder.appName('02f_splink_luis').getOrCreate()

print('DATA_DIR:', DATA_DIR)
print('modelo:', MODEL_JSON)
print('censo:', CENSO_CSV)
print('cpf:  ', CPF_CSV)
print('saida:', OUT_PARQUET)


In [ ]:
df_censo = spark.read.csv(str(CENSO_CSV), header=False, schema=schema)
df_censo = df_censo.withColumn(
    'unique_id', F.concat(F.lit('censo_'), F.col('id_seq'))
)
df_censo = df_censo.withColumn('idade', F.col('idade').cast(IntegerType()))

df_cpf = spark.read.csv(str(CPF_CSV), header=False, schema=schema)
df_cpf = df_cpf.withColumn(
    'unique_id', F.concat(F.lit('cpf_'), F.col('id_seq'))
)
df_cpf = df_cpf.withColumn('idade', F.col('idade').cast(IntegerType()))

print('censo:', df_censo.count())
print('cpf:  ', df_cpf.count())
df_censo.select(
    'unique_id', 'nome_completo_phon', 'data_nascimento', 'idade'
).show(3, truncate=False)
df_cpf.select(
    'unique_id', 'nome_completo_phon', 'data_nascimento', 'idade'
).show(3, truncate=False)


In [ ]:
from splink import Linker, block_on
from splink.backends.spark import SparkAPI

blocking_rules = [
    block_on('nome_completo_phon'),
    block_on('primeiro_nome_phon', 'ultimo_nome_phon', 'ano_nascimento'),
    block_on('primeiro_nome_phon', 'ultimo_nome_phon', 'mes_nascimento', 'dia_nascimento'),
    block_on('primeiro_nome_phon', 'ultimo_nome_phon', 'mes_nascimento', 'ano_nascimento'),
    block_on('primeiro_nome_phon', 'data_nascimento'),
    block_on('ultimo_nome_phon', 'data_nascimento'),
    block_on('primeiro_nome_phon', 'mes_nascimento', 'dia_nascimento', 'cod_municipio'),
    block_on('primeiro_nome_phon', 'mes_nascimento', 'ano_nascimento', 'cod_municipio'),
    block_on('ultimo_nome_phon', 'mes_nascimento', 'dia_nascimento', 'sexo', 'cep'),
    block_on('ultimo_nome_phon', 'mes_nascimento', 'ano_nascimento', 'sexo', 'cep'),
    (
        "l.ultimo_nome_phon = r.ultimo_nome_phon "
        "AND l.ano_nascimento = r.ano_nascimento "
        "AND l.mes_nascimento = r.mes_nascimento "
        "AND l.sexo = r.sexo "
        "AND damerau_levenshtein(l.primeiro_nome_phon, r.primeiro_nome_phon) <= 2 "
        "AND least(len(l.primeiro_nome_phon), len(r.primeiro_nome_phon)) >= 6 "
        "AND damerau_levenshtein(l.primeiro_nome_phon, r.primeiro_nome_phon) * 6 "
        "<= least(len(l.primeiro_nome_phon), len(r.primeiro_nome_phon))"
    ),
    (
        "l.primeiro_nome_phon = r.primeiro_nome_phon "
        "AND l.ultimo_nome_phon = r.ultimo_nome_phon "
        "AND l.cod_municipio = r.cod_municipio "
        "AND l.sexo = r.sexo "
        "AND abs(l.idade - r.idade) <= 1"
    ),
    (
        "l.logradouro_norm = r.logradouro_norm "
        "AND l.cep = r.cep "
        "AND l.cod_municipio = r.cod_municipio "
        "AND l.sexo = r.sexo "
        "AND abs(l.idade - r.idade) <= 1"
    ),
    (
        "l.nome_mae_phon = r.nome_mae_phon "
        "AND l.cod_municipio = r.cod_municipio "
        "AND l.sexo = r.sexo "
        "AND abs(l.idade - r.idade) <= 1"
    ),
]

print('regras de blocking:', len(blocking_rules))


In [ ]:
with open(MODEL_JSON) as f:
    settings = json.load(f)

settings['retain_intermediate_calculation_columns'] = False
settings['retain_matching_columns'] = False
settings['blocking_rules_to_generate_predictions'] = blocking_rules

db_api = SparkAPI(spark_session=spark)
linker = Linker(
    [df_censo, df_cpf],
    settings,
    db_api,
    input_table_aliases=['censo', 'cpf'],
)
print('Linker pronto:', MODEL_JSON)


In [ ]:
df_predict = linker.inference.predict(threshold_match_probability=0.05)
pred = df_predict.as_spark_dataframe()
print('predictions:', pred.count())
pred.select('unique_id_l', 'unique_id_r', 'match_probability', 'match_weight').show(5)


In [ ]:
keep = [
    c for c in (
        'unique_id_l', 'unique_id_r', 'match_probability', 'match_weight'
    )
    if c in pred.columns
]

(
    pred.select(*keep)
    .write.mode('overwrite')
    .parquet(str(OUT_PARQUET))
)

print('Predictions:', OUT_PARQUET)
print('colunas:', keep)
